# CRM Customer Cleansing — หาลูกค้าจริงที่ไม่ซ้ำ (ขั้นที่ 1 ของ SSOT Customer)

**เป้าหมาย:** เช็กซ้ำ*ภายใน* CRM ก่อนรวมกับฐานอื่น · ได้จำนวน "ลูกค้าจริงที่ไม่ซ้ำ" แยก 3 ชุด

| ชุด | ตาราง | ความหมาย |
|---|---|---|
| **CM** | `customer_master` | ลูกค้าทั้งหมด (ทุกคนที่เคยมีชื่อในบิล · เป็นสมาชิกหรือไม่ก็ได้) |
| **MB** | `members` | สมาชิก CRM / 7-Club |
| **MA** | `member_auth` ⨝ `members` | สมาชิกที่ยืนยันตัวตน / login 7Club+ (ชื่อดึงจาก `members`) |

**ชั้นของการ clean (ทำเหมือนกันทั้ง 3 ชุด)**

| Layer | ทำอะไร | ตัดแถวออกไหม |
|---|---|---|
| L0 Extract | ดึงข้อมูลทีละ 50,000 แถวตาม `id` · เลือกเฉพาะคอลัมน์ที่ใช้ | – |
| L1 Standardize | ตัดคำนำหน้า · แยกชื่อ-นามสกุลที่อยู่ช่องเดียว · แยกภาษาไทย/อังกฤษ · normalize เบอร์ · แก้โดเมนอีเมลที่สะกดผิด · แก้ปี พ.ศ. | ไม่ตัด (แก้ค่า) |
| L2 Validate contact | เบอร์ผิดรูป/เบอร์ปลอม · อีเมลผิดรูป/อีเมลปลอม → ใช้เป็นคีย์ไม่ได้ | ไม่ตัด (ค่ากลายเป็นว่าง) |
| L3 Classify | ไม่มีข้อมูลระบุตัว · บัญชีร้าน/สาขา · นิติบุคคล · ชื่อปลอม/placeholder · พนักงาน · เบอร์/อีเมลกลางที่ใช้ร่วมกันหลายคน | **แยกออกจากชุดบุคคล** |
| L4 Dedup | กฎจับคู่ R1–R7 (union-find) · ไทย/อังกฤษ + เบอร์/อีเมล · ข้ามภาษา · ชื่อ+วันเกิด | **รวมแถวซ้ำ** |
| L5 Review | ชื่อ-นามสกุลเดียวกันแต่คนละเบอร์ → ยังไม่รวม ส่งตรวจ | – |

**กติกาความปลอดภัย**
- CRM เป็น read-only · ดึงครั้งละ 50,000 แถว · ไม่ใช้ `SELECT *` · ไม่แตะคอลัมน์ที่ถูกซ่อน (`citizen_no`)
- ชื่อ เบอร์ อีเมล ถูกแปลงเป็น hash ในหน่วยความจำ · ไม่เขียน PII ลงดิสก์
- ตัวอย่างในโน้ตบุ๊กนี้ **mask ทุกค่า** (ชื่อเหลือ 2 ตัวแรก · เบอร์เหลือ 3 ตัวหน้า 2 ตัวท้าย)
- ไฟล์ที่รันแล้ว (มี output) อยู่ใน `scripts/crm/_out/` ซึ่งไม่ขึ้น git

In [ ]:
import re, sys, json, time, collections, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('future.infer_string', False)   # ใช้ Python re (รองรับ backreference \1) แทน Arrow regex
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

sys.path.insert(0, r'C:\Projects\my-first-project\scripts')
import db   # credential อ่านจาก env CRM_USER / CRM_PWD · ต้องเปิด VPN

CHUNK    = 50_000    # กติกา CRM: ไม่เกิน 50k แถวต่อ query
MAX_ROWS = None      # ใส่ตัวเลข (เช่น 300_000) เพื่อทดสอบเร็ว · None = ทั้งตาราง
pd.set_option('display.max_colwidth', 60)
cn = db.connect('crm', database='crm_pool_prod', timeout=600)
print('connected')

## ตัวช่วย mask สำหรับแสดงตัวอย่าง
ทุกตัวอย่างที่แสดงผ่าน `mask_*` — เห็นรูปแบบปัญหาได้ แต่ระบุตัวคนไม่ได้

In [ ]:
def mask_tok(s, keep=2):
    s = '' if s is None else str(s)
    return ' '.join(t[:keep] + '•' * min(max(len(t) - keep, 0), 4) for t in s.split(' ')) if s else ''
def mask_phone(s):
    s = '' if s is None else str(s)
    return s[:3] + 'x' * max(len(s) - 5, 0) + s[-2:] if len(s) > 5 else ('x' * len(s))
def mask_email(s):
    s = '' if s is None else str(s)
    if '@' not in s: return mask_tok(s)
    a, d = s.rsplit('@', 1)
    return a[:2] + '•••@' + d
def mask_digits(s):
    return re.sub(r'(?<=\d{3})\d', 'x', str(s))

print(mask_tok('สมชาย ใจดี'), '|', mask_phone('0812345678'), '|', mask_email('somchai.j@gmail.com'))

## L1–L3 · กฎทั้งหมด (dictionary + regex)

**คำนำหน้า** — ตัดออกจากต้น `first_name` เก็บแยกเป็น `title`
- `คุณ` / `นาย` / `นาง` ตัดเฉพาะเมื่อตามด้วย*พยัญชนะ*หรือช่องว่าง → ไม่ตัดชื่อจริงอย่าง *คุณากร* *คุณัญญา* *นายิกา*

**ชื่อ placeholder / ชื่อปลอม** (มีข้อใดข้อหนึ่ง = ไม่ใช่ชื่อคนจริง)
| เหตุผล | ตัวอย่างรูปแบบ |
|---|---|
| ว่าง / มีแต่คำนำหน้า | `''` · `คุณ` |
| placeholder | ลูกค้า · ลูกค้าทั่วไป · ไม่ระบุ · test · customer · `-` · `.` · xxx |
| มีตัวเลข | `0812…` (เบอร์ถูกกรอกในช่องชื่อ) · `test1` |
| ไม่มีตัวอักษรเลย | `???` · `..` |
| ตัวอักษรซ้ำ ≥ 3 ตัวติดกัน | `aaaa` · `กกกก` |
| แป้นพิมพ์มั่ว | `asdf` · `qwer` · `ฟหกด` |
| อังกฤษไม่มีสระเลย ≥ 4 ตัว | `sdfgh` · `bbnm` |
| ยาว 1 ตัวอักษร | `ก` · `A` |

**นิติบุคคล** — บริษัท · บจก · หจก · จำกัด · มหาชน · Co.,Ltd · Company · มูลนิธิ · โรงเรียน · มหาวิทยาลัย · โรงพยาบาล · ร้าน · สำนักงาน ฯลฯ หรือ `members.type = corporate`

**เบอร์โทร** — ตัดทุกอย่างที่ไม่ใช่ตัวเลข → `66…` เป็น `0…` → 9 หลักขึ้นต้น 6/8/9 เติม 0 → ใช้ได้เมื่อเป็นมือถือ `0[689]` 10 หลัก หรือเบอร์บ้าน `0[2-7]` 9 หลัก · เบอร์ปลอม = เลขซ้ำ ≥ 7 ตัว · `0123456789` · `0812345678`

**อีเมล** — lower + trim → แก้โดเมนสะกดผิด (`gmail.con` `gamil.com` `hotmial.com` …) → รูปแบบต้องถูก → อีเมลปลอม (`no@` `none@` `test@` `xxx@` · `example.com`)

In [ ]:
TH_CONS = '\u0E01-\u0E2E'
TITLE = re.compile(
    r'^\s*(นางสาว|น\.ส\.?|นส\.|เด็กชาย|เด็กหญิง|ด\.ช\.|ด\.ญ\.|ดร\.|'
    r'นาย(?=[\s' + TH_CONS + r'])|นาง(?=[\s' + TH_CONS + r'])|คุณ(?=[\s' + TH_CONS + r'])|'
    r'mrs\.|mr\.|ms\.|dr\.|(?:mrs|mr|ms|miss|dr|khun)(?=\s))\s*', re.I)

PLACEHOLDER = {'ลูกค้า','ลูกค้าทั่วไป','ลูกค้าเงินสด','คุณลูกค้า','ไม่ระบุ','ไม่ระบุชื่อ','ไม่มี','ไม่มีชื่อ','ไม่ทราบ',
               'ทดสอบ','สมาชิก','ชื่อ','test','tester','testing','customer','cash','walkin','guest','unknown',
               'none','null','na','n/a','no','noname','name','firstname','member','user','admin','-','--','---',
               '.','..','...','x','xx','xxx','xxxx','a','aa','abc','asd'}
PLACE_PREFIX = r'^(?:test|ทดสอบ|ลูกค้า|ไม่ระบุ|ไม่มีชื่อ|customer)'
KEYBOARD     = r'qwer|wert|asdf|sdfg|dfgh|fghj|ghjk|hjkl|zxcv|xcvb|ฟหกด|หกดเ|กดเ้|ๅ/-ภ|พะัี'
JURISTIC     = (r'บริษัท|บจก|บมจ|ห้างหุ้นส่วน|หจก|หสน|จำกัด|มหาชน|co\.?\s*,?\s*ltd|company|limited|\bltd\b|'
                r'corporation|\bcorp\b|\binc\b|มูลนิธิ|สมาคม|สหกรณ์|โรงเรียน|มหาวิทยาลัย|วิทยาลัย|โรงพยาบาล|'
                r'สำนักงาน|องค์การ|องค์กร|เทศบาล|กระทรวง|ร้าน|\bshop\b|\bstore\b|school|university|hospital|foundation')
DOMAIN_FIX = {**{d: 'gmail.com' for d in ['gmail.con','gmil.com','gamil.com','gmai.com','gnail.com','gmsil.com',
                                          'gmali.com','gmal.com','gmaol.com','gmail.co','gmail.cm','gmail.om',
                                          'gmaill.com','gmial.com','gmaik.com','gmail.comm','gmail.co.th','gmeil.com']},
              **{d: 'hotmail.com' for d in ['hotmail.con','hotmial.com','hotmai.com','hotmil.com','hotmal.com',
                                            'homail.com','hmail.com','hotmali.com','hotmail.cm']}}
EMAIL_OK     = r'^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$'
EMAIL_DUMMY  = r'^(?:no|none|noemail|nomail|test|x+|na|abc|a|-+|null|0+)@|@(?:example|test|xxx)\.com$'
INTERNAL     = r'\bID\d{2,}|บานาน่า|banana|studio\s?7|สตูดิโอ\s?7|comseven|คอมเซเว่น|\bbnn\b'
COMPANY_DOM  = r'@(?:[a-z0-9\-]+\.)*(?:comseven\.com|studio7thailand\.com)$'
PHONE_DUMMY  = r'(\d)\1{6,}|^0123456789$|^0812345678$|^0987654321$'

# ชั้นของตัวอักษรต้นชื่อ ไทย ↔ อังกฤษ (ใช้กับกฎข้ามภาษา R6)
TH2C = {**dict.fromkeys('กขฃคฅฆ','K'), 'ง':'N', **dict.fromkeys('จฉชฌ','J'), 'ซ':'S', 'ญ':'Y', 'ฎ':'D',
        **dict.fromkeys('ฏฐฑฒตถทธ','T'), 'ณ':'N', 'ด':'D', 'น':'N', 'บ':'B', **dict.fromkeys('ปผพภ','P'),
        **dict.fromkeys('ฝฟ','F'), 'ม':'M', 'ย':'Y', **dict.fromkeys('รฤ','R'), **dict.fromkeys('ลฦฬ','L'),
        'ว':'W', **dict.fromkeys('ศษส','S'), **dict.fromkeys('หฮ','H'), 'อ':'A'}
EN2C = {**dict.fromkeys('KGQ','K'), **dict.fromkeys('CJ','J'), **dict.fromkeys('SZX','S'), 'D':'D', 'T':'T',
        'B':'B', 'P':'P', 'F':'F', 'M':'M', 'N':'N', 'Y':'Y', 'R':'R', 'L':'L', **dict.fromkeys('WV','W'),
        'H':'H', **dict.fromkeys('AEIOU','A')}
TH_FIRST = r'^[เแโใไ]?(?:ห(?=[งญนมยรลว]))?([ก-ฮ])'   # ข้ามสระหน้า + ห นำ
CLS = {c: i + 1 for i, c in enumerate('KJSDTBPFMNYRLWHA')}
print('rules ready')

### ทดลองกฎกับชื่อสมมติ (ไม่ใช่ข้อมูลลูกค้าจริง)
ฟังก์ชัน `clean_names` ใช้กับทั้งตารางแบบ vectorized — ลองกับตัวอย่างสมมติก่อนเพื่อดูว่าแต่ละกฎทำอะไร

In [ ]:
def clean_names(first, last, title_col=None):
    f0 = first.fillna('').astype(str); l0 = last.fillna('').astype(str)
    f = f0.str.replace(r'\s+', ' ', regex=True).str.strip()
    l = l0.str.replace(r'\s+', ' ', regex=True).str.strip()
    t = f.str.extract(TITLE, expand=False)
    f = f.str.replace(TITLE, '', regex=True)
    t2 = f.str.extract(TITLE, expand=False)                  # คำนำหน้าซ้อน เช่น 'คุณ นาย'
    f = f.str.replace(TITLE, '', regex=True).str.strip()
    title = t.fillna(t2)
    title_in = title.notna()                                  # คำนำหน้าที่อยู่ในช่องชื่อจริงๆ
    if title_col is not None:
        title = title.fillna(title_col.replace('', np.nan))
    # ชื่อ+นามสกุลอยู่ช่องเดียว
    split = (l == '') & f.str.contains(' ', regex=False)
    if split.any():
        p = f[split].str.split(' ', n=1)
        f.loc[split] = p.str[0]; l.loc[split] = p.str[1]
    # first_name ลงท้ายด้วยนามสกุลซ้ำ
    dupl = (l != '') & pd.Series([a.endswith(' ' + b) for a, b in zip(f, l)], index=f.index)
    if dupl.any():
        f.loc[dupl] = [a[:-len(b) - 1] for a, b in zip(f[dupl], l[dupl])]
    middle = f.str.contains(' ', regex=False)
    th = f.str.contains('[\u0E01-\u0E4F]'); en = f.str.contains('[A-Za-z]')
    lang = np.select([th & ~en, en & ~th, th & en], [1, 2, 3], 0).astype('u1')
    lth = l.str.contains('[\u0E01-\u0E4F]'); len_ = l.str.contains('[A-Za-z]')
    key = (f + l).str.replace(r'[\s.\-]', '', regex=True).str.lower()
    fk = f.str.replace(r'[\s.\-]', '', regex=True).str.lower()
    # ชื่อปลอม — เหตุผลแรกที่เจอ
    reason = np.select(
        [fk == '',
         fk.isin(PLACEHOLDER) | fk.str.contains(PLACE_PREFIX),
         f.str.contains(r'[0-9๐-๙]'),
         ~f.str.contains('[A-Za-z\u0E01-\u0E2E]'),
         fk.str.contains(r'(.)\1\1'),
         fk.str.contains(KEYBOARD),
         (lang == 2) & fk.str.fullmatch(r'[bcdfghjklmnpqrstvwxz]{4,}'),
         fk.str.len() < 2],
        ['empty/title-only', 'placeholder', 'digit', 'no-letter', 'repeat-char', 'keyboard', 'no-vowel', 'too-short'],
        '')
    full = (f + ' ' + l).str.strip()
    return dict(first=f, last=l, title=title, title_in=title_in, split=split, dup_last=dupl, middle=middle, lang=lang,
                last_lang=np.select([lth & ~len_, len_ & ~lth], [1, 2], 0), key=key, junk=pd.Series(reason, index=f.index),
                juristic=full.str.contains(JURISTIC, case=False), internal=full.str.contains(INTERNAL, case=False), same_fl=(fk != '') & (fk == l.str.replace(r'[\s.\-]', '', regex=True).str.lower()))

demo = pd.DataFrame({'first_name': ['คุณสมชาย', 'นางสาวสุดา ใจงาม', 'คุณากร', 'Mr.Somchai', 'ลูกค้าทั่วไป', 'asdfg',
                                    'สมหญิง ศรีสุข', 'บริษัท ตัวอย่าง', '0812345678', 'aaaa', 'คุณ'],
                     'last_name':  ['ใจดี', '', 'มั่นคง', 'Jaidee', '', '', 'ศรีสุข', 'จำกัด', '', '', '']})
r = clean_names(demo.first_name, demo.last_name)
pd.DataFrame({'ก่อน (first | last)': demo.first_name + ' | ' + demo.last_name, 'title': r['title'],
              'first': r['first'], 'last': r['last'], 'lang': r['lang'], 'ชื่อปลอม': r['junk'], 'นิติบุคคล': r['juristic']})

In [ ]:
def clean_phone(s):
    d = s.fillna('').astype(str).str.replace(r'\D', '', regex=True)
    had_sym = s.fillna('').astype(str).str.contains(r'\D') & (d != '')
    m66 = (d.str.len() == 11) & d.str.startswith('66'); d = d.where(~m66, '0' + d.str[2:])
    m0 = (d.str.len() == 9) & d.str[:1].isin(list('689')); d = d.where(~m0, '0' + d)
    ok = d.str.fullmatch(r'0[689]\d{8}') | d.str.fullmatch(r'0[2-7]\d{7}')
    dummy = ok & d.str.contains(PHONE_DUMMY)
    status = np.select([d == '', dummy, ~ok], ['empty', 'dummy', 'invalid'], 'ok')
    val = np.where(status == 'ok', pd.to_numeric(d.where(status == 'ok', '0'), errors='coerce'), 0).astype('i8')
    return dict(norm=d, status=pd.Series(status, index=s.index), fixed=(had_sym | m66 | m0) & (status == 'ok'), val=val)

def clean_email(s):
    e = s.fillna('').astype(str).str.strip().str.lower()
    dom = e.str.rsplit('@', n=1).str[-1].where(e.str.contains('@'), '')
    fix = dom.isin(DOMAIN_FIX.keys())
    e = e.where(~fix, e.str.rsplit('@', n=1).str[0] + '@' + dom.map(DOMAIN_FIX))
    ok = e.str.fullmatch(EMAIL_OK)
    dummy = ok & e.str.contains(EMAIL_DUMMY)
    status = np.select([e == '', dummy, ~ok], ['empty', 'dummy', 'invalid'], 'ok')
    good = status == 'ok'
    h = np.where(good, pd.util.hash_array(e.to_numpy(object)), 0).astype('u8')
    return dict(norm=e, status=pd.Series(status, index=s.index), fixed_domain=fix & good,
                company=good & e.str.contains(COMPANY_DOM), hash=h)

def clean_birth(s):
    t = s.astype(str).str[:10]
    y = pd.to_numeric(t.str[:4], errors='coerce'); m = pd.to_numeric(t.str[5:7], errors='coerce'); d = pd.to_numeric(t.str[8:10], errors='coerce')
    be = y > 2400; y = y.where(~be, y - 543)                       # ปี พ.ศ. → ค.ศ.
    ok = y.between(1920, 2026) & m.between(1, 12) & d.between(1, 31)
    jan1 = ok & (m == 1) & (d == 1)                                # น่าจะเป็นค่า default
    val = np.where(ok & ~jan1, (y * 10000 + m * 100 + d).fillna(0), 0).astype('i4')
    present = s.notna() & ~t.isin(['None', 'NaT', '0000-00-00', 'nan', ''])
    return dict(val=val, be_fixed=be & ok, invalid=present & ~ok, jan1=jan1)

def init_class(first, last, lang):
    def th(x): return x.str.extract(TH_FIRST, expand=False).map(TH2C)
    def en(x): return x.str[:1].str.upper().map(EN2C)
    a = np.where(lang == 1, th(first), np.where(lang == 2, en(first), None))
    b = np.where(lang == 1, th(last), np.where(lang == 2, en(last), None))
    a = pd.Series(a).map(CLS).fillna(0).astype('u2').to_numpy(); b = pd.Series(b).map(CLS).fillna(0).astype('u2').to_numpy()
    return (a * 32 + b).astype('u2')

ph = clean_phone(pd.Series(['081-234-5679', '+66 81 234 5679', '812345679', '0999999999', '12345', None]))
em = clean_email(pd.Series(['Somchai@GMAIL.CON', 'no@gmail.com', 'abc', 'staff@comseven.com', None]))
display(pd.DataFrame({'phone ก่อน': ['081-234-5679', '+66 81 234 5679', '812345679', '0999999999', '12345', None],
                      'หลัง': ph['norm'], 'สถานะ': ph['status']}))
pd.DataFrame({'email ก่อน': ['Somchai@GMAIL.CON', 'no@gmail.com', 'abc', 'staff@comseven.com', None],
              'หลัง': em['norm'], 'สถานะ': em['status'], 'โดเมนพนักงาน': em['company']})

## L0 · Extract — ดึงทีละ 50,000 แถว แล้ว clean ทันที
ในแต่ละ chunk: clean ชื่อ/เบอร์/อีเมล/วันเกิด → เก็บเฉพาะ **hash + ธง** ลง array (ไม่เก็บชื่อจริง) · นับสถิติแต่ละกฎ · เก็บตัวอย่างแบบ mask ไว้กฎละไม่กี่แถว

In [ ]:
SETS = {
  'CM': dict(name='customer_master · ลูกค้าทั้งหมด', table='customer_master',
             sql="SELECT id, first_name, last_name, email, phone, birth_date, is_staff "
                 "FROM customer_master WHERE id BETWEEN %s AND %s"),
  'MB': dict(name='members · สมาชิก CRM', table='members',
             sql="SELECT id, customer_master_id AS link_id, type, title, first_name, last_name, email, phone, birth_date "
                 "FROM members WHERE id BETWEEN %s AND %s"),
  'MA': dict(name='member_auth · ยืนยันตัวตน 7Club+', table='member_auth',
             sql="SELECT ma.id, ma.member_id AS link_id, ma.identity_no, ma.email, ma.phone, m.type, m.title, m.first_name, m.last_name, m.birth_date "
                 "FROM member_auth ma LEFT JOIN members m ON m.id = ma.member_id WHERE ma.id BETWEEN %s AND %s"),
}

def thai_id_ok(s):
    d = s.fillna('').astype(str).str.replace(r'\D', '', regex=True)
    ok = d.str.len() == 13
    out = np.zeros(len(d), bool)
    if ok.any():
        a = np.array([list(map(int, x)) for x in d[ok]])
        chk = (11 - (a[:, :12] * np.arange(13, 1, -1)).sum(1) % 11) % 10
        out[np.flatnonzero(ok.to_numpy())] = chk == a[:, 12]
    return d, out

EX_N = 6
def add_ex(ex, k, rows):
    lst = ex.setdefault(k, [])
    if len(lst) < EX_N: lst.extend(rows[:EX_N - len(lst)])

def run_set(code_):
    global cn
    S = SETS[code_]; cur = cn.cursor()
    cur.execute(f"SELECT MIN(id), MAX(id) FROM {S['table']}"); lo, hi = cur.fetchone()
    cnt = collections.Counter(); ex = {}; junk_vals = collections.Counter()
    t0 = time.time(); rows = 0
    size = hi - lo + 1           # array จองตามช่วง id · ตำแหน่ง = id - lo · id ที่ไม่มีแถว status = 9
    arr = dict(id=np.arange(lo, hi + 1, dtype='u4'), status=np.full(size, 9, 'u1'), lang=np.zeros(size, 'u1'),
               name=np.zeros(size, 'u8'), phone=np.zeros(size, 'i8'), email=np.zeros(size, 'u8'), birth=np.zeros(size, 'i4'),
               init=np.zeros(size, 'u2'), company=np.zeros(size, bool), staff=np.zeros(size, 'u1'), link=np.zeros(size, 'u8'))
    if code_ == 'MA': arr['idno'] = np.zeros(size, 'u8')
    base = lo
    for start in range(lo, hi + 1, CHUNK):
        for attempt in range(6):                        # VPN/Aurora หลุด → ต่อใหม่แล้วดึง chunk เดิมซ้ำ
            try:
                cur.execute(S['sql'], (start, start + CHUNK - 1)); recs = cur.fetchall(); break
            except Exception as err:
                print(f'  retry {attempt + 1} @ id {start:,}: {str(err)[:80]}'); time.sleep(30 * (attempt + 1))
                try: cn.close()
                except Exception: pass
                cn = db.connect('crm', database='crm_pool_prod', timeout=600); cur = cn.cursor()
        else:
            raise RuntimeError(f'ดึง chunk id {start:,} ไม่ได้หลังลอง 6 ครั้ง')
        if not recs: continue
        cols = [c[0] for c in cur.description]
        df = pd.DataFrame(recs, columns=cols); rows += len(df)
        nm = clean_names(df.first_name, df.last_name, df['title'].fillna('').astype(str) if 'title' in df else None)
        ph = clean_phone(df.phone); em = clean_email(df.email); bd = clean_birth(df.birth_date)
        jur = nm['juristic'].to_numpy() | (df['type'].eq('corporate').to_numpy() if 'type' in df else False)
        junk = (nm['junk'] != '').to_numpy()
        noid = junk & (ph['status'] != 'ok').to_numpy() & (em['status'] != 'ok').to_numpy()
        intl = nm['internal'].to_numpy()
        status = np.select([noid, intl, jur, junk], [1, 4, 2, 3], 0).astype('u1')   # 0=บุคคล 1=ไม่มีข้อมูลระบุตัว 4=บัญชีร้าน/สาขา 2=นิติบุคคล 3=ชื่อปลอม
        pos = df.id.to_numpy('i8') - base
        if 'identity_no' in df:
            idd, idok = thai_id_ok(df.identity_no)
            arr['idno'][pos] = np.where(idok, pd.util.hash_array(idd.to_numpy(object)), 0).astype('u8')
            cnt['id_present'] += int((idd != '').sum()); cnt['id_valid'] += int(idok.sum())
        arr['status'][pos] = status; arr['lang'][pos] = nm['lang']
        arr['name'][pos] = np.where(junk, 0, pd.util.hash_array(nm['key'].to_numpy(object))).astype('u8')
        arr['phone'][pos] = ph['val']; arr['email'][pos] = em['hash']; arr['birth'][pos] = bd['val']
        arr['init'][pos] = init_class(nm['first'], nm['last'], nm['lang'])
        arr['company'][pos] = em['company'].to_numpy()
        if 'link_id' in df: arr['link'][pos] = df['link_id'].fillna(0).astype('int64').to_numpy('u8')
        if 'is_staff' in df: arr['staff'][pos] = df['is_staff'].fillna(0).astype(int).to_numpy('u1')
        # ── counters ──
        cnt['rows'] += len(df)
        cnt['title'] += int(nm['title_in'].sum()); cnt['split'] += int(nm['split'].sum())
        cnt['dup_last'] += int(nm['dup_last'].sum()); cnt['middle'] += int(nm['middle'].sum())
        for k, v in zip(*np.unique(nm['lang'], return_counts=True)): cnt[f'lang_{k}'] += int(v)
        cnt['th_first_en_last'] += int(((nm['lang'] == 1) & (nm['last_lang'] == 2)).sum())
        cnt['en_first_th_last'] += int(((nm['lang'] == 2) & (nm['last_lang'] == 1)).sum())
        cnt['same_first_last'] += int(nm['same_fl'].sum())
        for k, v in nm['junk'][nm['junk'] != ''].value_counts().items(): cnt[f'junk_{k}'] += int(v)
        for k, v in ph['status'].value_counts().items(): cnt[f'phone_{k}'] += int(v)
        cnt['phone_fixed'] += int(ph['fixed'].sum())
        for k, v in em['status'].value_counts().items(): cnt[f'email_{k}'] += int(v)
        cnt['email_domain_fixed'] += int(em['fixed_domain'].sum()); cnt['email_company'] += int(em['company'].sum())
        cnt['birth_present'] += int((bd['val'] > 0).sum()); cnt['birth_be_fixed'] += int(bd['be_fixed'].sum())
        cnt['birth_invalid'] += int(bd['invalid'].sum()); cnt['birth_jan1'] += int(bd['jan1'].sum())
        for k, v in zip(*np.unique(status, return_counts=True)): cnt[f'status_{k}'] += int(v)
        if 'is_staff' in df: cnt['is_staff'] += int(df['is_staff'].fillna(0).astype(int).sum())
        junk_vals.update(mask_digits(x)[:30] for x in (df.first_name.fillna('').astype(str).str.strip() + ' ' + df.last_name.fillna('').astype(str).str.strip()).str.strip()[junk])
        # ── examples (masked) ──
        fo, lo_ = df.first_name.fillna('').astype(str), df.last_name.fillna('').astype(str)
        m = nm['title'].notna()
        add_ex(ex, 'ตัดคำนำหน้า', [(nm['title'][i] + mask_tok(nm['first'][i]) + ' | ' + mask_tok(lo_[i]), mask_tok(nm['first'][i]), str(nm['title'][i])) for i in df.index[m][:EX_N]])
        m = nm['split']
        add_ex(ex, 'แยกชื่อ-นามสกุลจากช่องเดียว', [(mask_tok(fo[i]) + ' | (ว่าง)', mask_tok(nm['first'][i]) + ' | ' + mask_tok(nm['last'][i]), '') for i in df.index[m][:EX_N]])
        m = nm['dup_last']
        add_ex(ex, 'นามสกุลซ้ำในช่องชื่อ', [(mask_tok(fo[i]) + ' | ' + mask_tok(lo_[i]), mask_tok(nm['first'][i]) + ' | ' + mask_tok(nm['last'][i]), '') for i in df.index[m][:EX_N]])
        m = (nm['lang'] == 1) & (nm['last_lang'] == 2)
        add_ex(ex, 'ชื่อไทย + นามสกุลอังกฤษ', [(mask_tok(nm['first'][i]) + ' | ' + mask_tok(nm['last'][i]), 'TH | EN', '') for i in df.index[m][:EX_N]])
        m = ph['fixed']
        add_ex(ex, 'แก้รูปแบบเบอร์', [(mask_digits(re.sub(r'(?<=.{3}).(?=.{2})', 'x', str(df.phone[i]))), mask_phone(ph['norm'][i]), '') for i in df.index[m][:EX_N]])
        m = em['fixed_domain']
        add_ex(ex, 'แก้โดเมนอีเมล', [(mask_email(str(df.email[i]).strip().lower()), mask_email(em['norm'][i]), '') for i in df.index[m][:EX_N]])
        m = status == 4
        add_ex(ex, 'บัญชีร้าน/สาขา', [(mask_tok(fo[i] + ' ' + lo_[i], keep=6), '→ แยกออก (ไม่ใช่ลูกค้า)', '') for i in df.index[m][:EX_N]])
        m = jur & (status == 2)
        add_ex(ex, 'นิติบุคคล', [(mask_tok(fo[i] + ' ' + lo_[i], keep=6), '→ ชุดนิติบุคคล', '') for i in df.index[m][:EX_N]])
        if MAX_ROWS and rows >= MAX_ROWS: break
    print(f"{code_}: {rows:,} rows · {time.time() - t0:,.0f}s")
    return dict(cnt=cnt, ex=ex, junk_vals=junk_vals, arr=arr)

## L3 · เบอร์/อีเมลที่ใช้ร่วมกันหลายคน และอีเมลพนักงาน (@comseven.com)
- **เบอร์หรืออีเมลเดียวที่มีชื่อต่างกัน ≥ 5 ชื่อ** = เบอร์ร้าน / พนักงานกรอกแทนลูกค้า → *ไม่ใช้เป็นคีย์จับคู่* (แถวลูกค้ายังอยู่)
- **อีเมล @comseven.com**
  - 1 อีเมล = 1 ชื่อ → น่าจะเป็นพนักงานที่ซื้อของเอง = **ลูกค้าจริง** เก็บไว้ ติดธง `is_employee`
  - 1 อีเมล ≥ 2 ชื่อ → พนักงานใส่อีเมลตัวเองให้ลูกค้าที่ไม่มีอีเมล → ลบอีเมลออก (ลูกค้ายังอยู่)

In [ ]:
def distinct_per_key(key, name, mask):
    # จำนวนชื่อไม่ซ้ำต่อคีย์ (vectorized ด้วย sort) · คืนค่าต่อแถว
    idx = np.flatnonzero(mask & (key != 0) & (name != 0))
    out = np.zeros(len(key), 'u4')
    if len(idx) == 0: return out
    k, n = key[idx], name[idx]
    o = np.lexsort((n, k)); k, n, ii = k[o], n[o], idx[o]
    newk = np.r_[True, k[1:] != k[:-1]]; newn = newk | np.r_[True, n[1:] != n[:-1]]
    grp = np.cumsum(newk) - 1
    nd = np.bincount(grp, weights=newn).astype('u4')
    out[ii] = nd[grp]
    return out

def shared_contacts(R, th=5):
    a = R['arr']; c = R['cnt']; person = a['status'] == 0
    dp = distinct_per_key(a['phone'], a['name'], person)
    de = distinct_per_key(a['email'], a['name'], person)
    shp = dp >= th; she = de >= th
    c['shared_phone_rows'] = int(shp.sum()); c['shared_phone_keys'] = int(len(np.unique(a['phone'][shp])))
    c['shared_email_rows'] = int(she.sum()); c['shared_email_keys'] = int(len(np.unique(a['email'][she])))
    comp = a['company'] & person
    c['company_rows'] = int(comp.sum())
    c['company_1name'] = int((comp & (de == 1)).sum()); c['company_multi'] = int((comp & (de >= 2)).sum())
    a['employee'] = comp & (de == 1) | (a['staff'] > 0)
    a['phone_m'] = np.where(shp, 0, a['phone']); a['email_m'] = np.where(she | (comp & (de >= 2)), 0, a['email'])
    dist = collections.Counter(np.minimum(de[comp], 5)); c['company_dist'] = {int(k): int(v) for k, v in sorted(dist.items())}
    return R

## L4 · Dedup — กฎจับคู่ทีละขั้น (union-find)
แต่ละกฎสร้าง "เส้นเชื่อม" ระหว่างแถวที่มีคีย์ตรงกัน แล้วรวมเป็นกลุ่ม (connected components) — นับจำนวนลูกค้าไม่ซ้ำหลังเพิ่มกฎแต่ละข้อ

| กฎ | คีย์ | ความมั่นใจ |
|---|---|---|
| R0 | ชี้ไป record เดียวกัน (MB: `customer_master_id` ซ้ำ · MA: `member_id` ซ้ำ) | แน่นอน |
| R1 | เลขบัตรประชาชน (ผ่าน checksum) — เฉพาะ MA | สูงสุด |
| R2 | ชื่อไทย + เบอร์ | สูง |
| R3 | ชื่อไทย + อีเมล | สูง |
| R4 | ชื่ออังกฤษ + เบอร์ | สูง |
| R5 | ชื่ออังกฤษ + อีเมล | สูง |
| R6 | **ข้ามภาษา**: แถวไทย ↔ แถวอังกฤษ ที่เบอร์หรืออีเมลเดียวกัน + อักษรต้นชื่อและนามสกุลตรงกัน (ส↔S · จ↔J/C · ก↔K/G …) และในกลุ่มนั้นมีชื่อไทย 1 แบบ อังกฤษ 1 แบบ | กลาง-สูง |
| R7 | ชื่อ + วันเกิด (ไม่นับ 1 ม.ค. ที่น่าจะเป็นค่า default) | กลาง-สูง |

*ชื่อ* = ชื่อ+นามสกุลหลังตัดคำนำหน้า ตัดช่องว่าง/จุด/ขีด และเป็นตัวพิมพ์เล็ก · เบอร์/อีเมลที่ใช้ร่วมกัน ≥ 5 ชื่อไม่ถูกใช้

In [ ]:
def mix(*ks):
    h = np.zeros(len(ks[0]), 'u8')
    with np.errstate(over='ignore'):
        for k in ks: h = (h ^ k.astype('u8')) * np.uint64(0x9E3779B97F4A7C15) + np.uint64(0x632BE59BD9B4E019)
    return h

def edges_by_key(key, valid, idx):
    sel = np.flatnonzero(valid)
    if len(sel) < 2: return np.empty((0, 2), 'i8')
    k = key[sel]; o = np.argsort(k, kind='stable'); k, s = k[o], sel[o]
    same = k[1:] == k[:-1]
    return np.c_[s[:-1][same], s[1:][same]]

def cross_lang_edges(a, contact, person):
    # แถว TH/EN ที่ contact เดียวกัน + init class เดียวกัน · กลุ่มต้องมีชื่อไทย 1 แบบ และอังกฤษ 1 แบบพอดี
    sel = np.flatnonzero(person & (contact != 0) & np.isin(a['lang'], [1, 2]) & (a['init'] % 32 > 0) & (a['init'] // 32 > 0))
    if len(sel) == 0: return np.empty((0, 2), 'i8')
    df = pd.DataFrame({'g': mix(contact[sel], a['init'][sel]), 'lang': a['lang'][sel], 'name': a['name'][sel], 'i': sel})
    st = df.groupby(['g', 'lang']).name.nunique().unstack(fill_value=0)
    ok = st.index[(st.get(1, 0) == 1) & (st.get(2, 0) == 1)]
    d = df[df.g.isin(ok)].sort_values('g')
    first = d.groupby('g').i.transform('first')
    m = d.i != first
    return np.c_[first[m].to_numpy(), d.i[m].to_numpy()]

def dedup(R):
    a = R['arr']; n = len(a['id']); person = a['status'] == 0
    P, E, B, N, L = a['phone_m'], a['email_m'], a['birth'], a['name'], a['lang']
    th = person & (L == 1); en = person & (L == 2); anyn = person & (N != 0)
    rules = []
    if (a['link'] != 0).any(): rules.append(('R0 ชี้ record เดียวกัน', lambda: edges_by_key(a['link'], person & (a['link'] != 0), None)))
    if 'idno' in a: rules.append(('R1 เลขบัตร', lambda: edges_by_key(a['idno'], person & (a['idno'] != 0), None)))
    rules += [('R2 ชื่อไทย+เบอร์',    lambda: edges_by_key(mix(N, P), th & (P != 0), None)),
              ('R3 ชื่อไทย+อีเมล',    lambda: edges_by_key(mix(N, E), th & (E != 0), None)),
              ('R4 ชื่ออังกฤษ+เบอร์', lambda: edges_by_key(mix(N, P), en & (P != 0), None)),
              ('R5 ชื่ออังกฤษ+อีเมล', lambda: edges_by_key(mix(N, E), en & (E != 0), None)),
              ('R6 ข้ามภาษา TH↔EN',  lambda: np.r_[cross_lang_edges(a, P, person), cross_lang_edges(a, E, person)]),
              ('R7 ชื่อ+วันเกิด',      lambda: edges_by_key(mix(N, B), anyn & (B != 0) & (L != 3), None))]
    pidx = np.flatnonzero(person); n_person = len(pidx)
    allE = [np.empty((0, 2), 'i8')]; steps = []; prev = n_person
    for name, fn in rules:
        e = fn(); allE.append(e); EE = np.concatenate(allE)
        g = coo_matrix((np.ones(len(EE), 'i1'), (EE[:, 0], EE[:, 1])), shape=(n, n))
        nc, lab = connected_components(g, directed=False)
        uniq = len(np.unique(lab[pidx]))
        steps.append(dict(rule=name, edges=int(len(e)), merged=int(prev - uniq), unique_after=int(uniq)))
        prev = uniq
    a['cluster'] = lab
    R['steps'] = steps; R['n_person'] = n_person; R['unique'] = prev
    R['edges_r6'] = allE[[s['rule'] for s in steps].index('R6 ข้ามภาษา TH↔EN') + 1]
    return R

def review_same_name(R):
    # L5: ชื่อ-นามสกุลเดียวกัน แต่อยู่คนละกลุ่ม (คนละเบอร์/ไม่มีอะไรยืนยัน) → ส่งตรวจ ไม่รวมอัตโนมัติ
    a = R['arr']; person = (a['status'] == 0) & (a['name'] != 0)
    df = pd.DataFrame({'n': a['name'][person], 'c': a['cluster'][person], 'p': a['phone_m'][person]})
    g = df.drop_duplicates(['n', 'c']).groupby('n').c.size()
    multi = g[g >= 2]
    R['review'] = dict(name_groups=int(len(multi)), clusters=int(multi.sum()),
                       groups_2_3=int(((multi >= 2) & (multi <= 3)).sum()), groups_4plus=int((multi >= 4).sum()),
                       extra_if_merged=int((multi - 1).sum()))
    return R

## รันทั้ง 3 ชุด

In [ ]:
RESULTS = {}
for k in ['MA', 'MB', 'CM']:           # เล็กไปใหญ่
    R = run_set(k); R = shared_contacts(R); R = dedup(R); R = review_same_name(R)
    RESULTS[k] = R
    R['arr'] = {'id': R['arr']['id']} if k == 'CM' else None     # คืนหน่วยความจำ · เก็บ id ไว้ดึงตัวอย่าง R6
    import gc; gc.collect()
    print('  unique:', f"{R['unique']:,}", '· steps:', [(s['rule'], s['merged']) for s in R['steps']])

## ผล L1 Standardize · L2 Validate · L3 Classify (ทั้ง 3 ชุด)

In [ ]:
LBL = [('rows', 'แถวทั้งหมด'),
       ('title', 'L1 มีคำนำหน้าในช่องชื่อ → ตัดออก'), ('split', 'L1 ชื่อ+นามสกุลอยู่ช่องเดียว → แยก'),
       ('dup_last', 'L1 นามสกุลซ้ำในช่องชื่อ → ตัด'), ('middle', 'L1 ยังมีช่องว่างในชื่อ (ชื่อกลาง/ชื่อยาว)'),
       ('lang_1', 'L1 ชื่อภาษาไทย'), ('lang_2', 'L1 ชื่อภาษาอังกฤษ'), ('lang_3', 'L1 ไทยปนอังกฤษในช่องเดียว'),
       ('th_first_en_last', 'L1 ชื่อไทย + นามสกุลอังกฤษ'), ('en_first_th_last', 'L1 ชื่ออังกฤษ + นามสกุลไทย'),
       ('phone_ok', 'L2 เบอร์ใช้ได้'), ('phone_fixed', 'L2   ↳ แก้รูปแบบแล้วใช้ได้'), ('phone_empty', 'L2 เบอร์ว่าง'),
       ('phone_invalid', 'L2 เบอร์ผิดรูป'), ('phone_dummy', 'L2 เบอร์ปลอม (เลขซ้ำ/เรียง)'),
       ('email_ok', 'L2 อีเมลใช้ได้'), ('email_domain_fixed', 'L2   ↳ แก้โดเมนสะกดผิด'), ('email_empty', 'L2 อีเมลว่าง'),
       ('email_invalid', 'L2 อีเมลผิดรูป'), ('email_dummy', 'L2 อีเมลปลอม'),
       ('birth_present', 'L2 วันเกิดใช้ได้'), ('birth_be_fixed', 'L2   ↳ แก้ปี พ.ศ.'), ('birth_jan1', 'L2 วันเกิด 1 ม.ค. (ไม่ใช้จับคู่)'),
       ('status_1', 'L3 ✂ ไม่มีข้อมูลระบุตัว (ชื่อใช้ไม่ได้ + ไม่มีเบอร์/อีเมล)'), ('status_4', 'L3 ✂ บัญชีร้าน/สาขา (ID สาขา · Banana · Studio7)'), ('status_2', 'L3 ✂ นิติบุคคล → แยกชุด'),
       ('status_3', 'L3 ✂ ชื่อปลอม/placeholder (มีเบอร์หรืออีเมล)'), ('status_0', 'L3 ✔ บุคคลที่ผ่านเข้า dedup'),
       ('same_first_last', 'L3 ⚑ ชื่อ = นามสกุล (ติดธง)'), ('is_staff', 'L3 ⚑ is_staff = 1'),
       ('email_company', 'L3 ⚑ อีเมล @comseven.com'), ('company_1name', 'L3   ↳ 1 อีเมล 1 ชื่อ = พนักงาน (เก็บเป็นลูกค้า)'),
       ('company_multi', 'L3   ↳ 1 อีเมลหลายชื่อ = กรอกแทนลูกค้า (ลบอีเมล)'),
       ('shared_phone_rows', 'L3 ⚑ แถวที่ใช้เบอร์กลาง (≥ 5 ชื่อ)'), ('shared_phone_keys', 'L3   ↳ จำนวนเบอร์กลาง'),
       ('shared_email_rows', 'L3 ⚑ แถวที่ใช้อีเมลกลาง (≥ 5 ชื่อ)'), ('shared_email_keys', 'L3   ↳ จำนวนอีเมลกลาง'),
       ('id_present', 'MA มีเลขบัตร'), ('id_valid', 'MA เลขบัตรผ่าน checksum')]
T = pd.DataFrame({k: [RESULTS[k]['cnt'].get(c, 0) for c, _ in LBL] for k in RESULTS}, index=[l for _, l in LBL])[['CM', 'MB', 'MA']]
T.style.format('{:,.0f}')

### เหตุผลที่ชื่อถูกมองว่าไม่ใช่ชื่อจริง (นับรวมทุกสถานะ)

In [ ]:
J = pd.DataFrame({k: {c[5:]: v for c, v in RESULTS[k]['cnt'].items() if c.startswith('junk_')} for k in RESULTS}).fillna(0).astype(int)[['CM', 'MB', 'MA']]
display(J.sort_values('CM', ascending=False).style.format('{:,.0f}'))
print('ค่าชื่อปลอมที่เจอบ่อยสุดใน CM (ตัวเลขถูก mask · แสดงเฉพาะค่าที่ซ้ำ ≥ 20 ครั้ง = ไม่ระบุตัวคน):')
pd.DataFrame([(v, c) for v, c in RESULTS['CM']['junk_vals'].most_common(40) if c >= 20], columns=['ค่า', 'แถว'])

### ตัวอย่างแต่ละกฎ (mask แล้ว)

In [ ]:
for k in ['CM', 'MB']:
    print(f"\n══ {k} ══")
    for rule, rows in RESULTS[k]['ex'].items():
        if rows:
            display(pd.DataFrame(rows, columns=['ก่อน', 'หลัง', 'title']).assign(กฎ=rule)[['กฎ', 'ก่อน', 'หลัง']])

### อีเมล @comseven.com — พนักงาน หรือ ลูกค้าที่ไม่มีอีเมล?
นับจำนวนชื่อไม่ซ้ำต่ออีเมลเดียว (5 = 5 ชื่อขึ้นไป)

In [ ]:
pd.DataFrame({k: RESULTS[k]['cnt'].get('company_dist', {}) for k in RESULTS}).fillna(0).astype(int).rename_axis('จำนวนชื่อต่ออีเมล')

## ผล L4 Dedup — ลูกค้าไม่ซ้ำหลังเพิ่มกฎแต่ละข้อ
`merged` = จำนวนแถวที่ถูกรวมเข้ากับแถวอื่นด้วยกฎนั้น (นับเพิ่มจากกฎก่อนหน้า)

In [ ]:
for k in ['CM', 'MB', 'MA']:
    R = RESULTS[k]
    print(f"\n{k} · {SETS[k]['name']} · บุคคลที่เข้า dedup {R['n_person']:,} แถว")
    display(pd.DataFrame(R['steps']).style.format({'edges': '{:,}', 'merged': '{:,}', 'unique_after': '{:,}'}))

### ตัวอย่างกฎข้ามภาษา R6 (ดึง 6 กลุ่มกลับมาดู · mask แล้ว)

In [ ]:
R = RESULTS['CM']; a = R['arr']; e = R['edges_r6']
if e is not None and len(e):
    pick = e[np.random.default_rng(7).choice(len(e), size=min(6, len(e)), replace=False)]
    ids = [int(x) for p in pick for x in a['id'][p]]
    cur = cn.cursor()
    cur.execute('SELECT id, first_name, last_name, phone, email FROM customer_master WHERE id IN (' + ','.join(['%s'] * len(ids)) + ')', ids)
    got = {r[0]: r for r in cur.fetchall()}
    rows = []
    for g, p in enumerate(pick):
        for x in a['id'][p]:
            r = got.get(int(x))
            if r: rows.append((g + 1, mask_tok(r[1]), mask_tok(r[2]), mask_phone(clean_phone(pd.Series([r[3]]))['norm'][0]), mask_email(r[4] or '')))
    display(pd.DataFrame(rows, columns=['กลุ่ม', 'ชื่อ', 'นามสกุล', 'เบอร์', 'อีเมล']))

## L5 · ชื่อ-นามสกุลเดียวกัน แต่คนละเบอร์ — ส่งตรวจ ไม่รวมอัตโนมัติ
ชื่อซ้ำที่ยังอยู่คนละกลุ่มหลัง R1–R7 = ไม่มีเบอร์/อีเมล/วันเกิดใดยืนยัน · อาจเป็นคนเดียวที่เปลี่ยนเบอร์ **หรือ** คนละคนชื่อเหมือนกัน
- 2–3 กลุ่มต่อชื่อ → มีโอกาสเป็นคนเดียว (เปลี่ยนเบอร์) → ใช้ข้อมูลฐานอื่น (ITEC บิล · K2/ITOS เลขบัตร) ช่วยยืนยันตอนรวมข้าม BU
- ≥ 4 กลุ่มต่อชื่อ → ชื่อโหล → ไม่รวม

In [ ]:
pd.DataFrame({k: RESULTS[k]['review'] for k in RESULTS})[['CM', 'MB', 'MA']].style.format('{:,.0f}')

## Funnel สรุป

In [ ]:
def funnel(k):
    R = RESULTS[k]; c = R['cnt']
    f = [('แถวดิบ', c['rows']), ('✂ ไม่มีข้อมูลระบุตัว', -c.get('status_1', 0)), ('✂ บัญชีร้าน/สาขา', -c.get('status_4', 0)), ('✂ นิติบุคคล (แยกชุด)', -c.get('status_2', 0)),
         ('✂ ชื่อปลอม/placeholder', -c.get('status_3', 0))]
    f += [('✂ ' + s['rule'], -s['merged']) for s in R['steps']]
    f += [('= ลูกค้าบุคคลไม่ซ้ำ', R['unique'])]
    return f
SUMMARY = {k: dict(name=SETS[k]['name'], funnel=funnel(k), cnt={a: b for a, b in RESULTS[k]['cnt'].items() if not isinstance(b, dict)},
                   company_dist=RESULTS[k]['cnt'].get('company_dist', {}), steps=RESULTS[k]['steps'], review=RESULTS[k]['review'],
                   n_person=RESULTS[k]['n_person'], unique=RESULTS[k]['unique'],
                   junk_top=[(v, c) for v, c in RESULTS[k]['junk_vals'].most_common(25) if c >= 50]) for k in RESULTS}
SUMMARY['_run'] = time.strftime('%Y-%m-%d %H:%M')
for k in ['CM', 'MB', 'MA']:
    print(f"\n{k} · {SETS[k]['name']}")
    for lab, v in SUMMARY[k]['funnel']: print(f"   {lab:<34}{v:>14,}")
import os
os.makedirs(r'C:\Projects\my-first-project\scripts\crm\_out', exist_ok=True)
with open(r'C:\Projects\my-first-project\scripts\crm\_out\cleansing_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(SUMMARY, fp, ensure_ascii=False, indent=1, default=int)
print('\nsaved summary (ตัวเลขรวมเท่านั้น ไม่มี PII)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Tahoma'
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, k in zip(axes, ['CM', 'MB', 'MA']):
    f = SUMMARY[k]['funnel']; labs = [x[0] for x in f]; vals = [x[1] for x in f]
    rem = np.cumsum(vals[:-1])                     # เหลือเท่าไรหลังแต่ละขั้น
    for i, v in enumerate(vals):
        if i in (0, len(vals) - 1):
            ax.barh(i, v, color='#1f4e79'); x = v
        else:
            ax.barh(i, -v, left=rem[i], color='#c0504d'); x = rem[i - 1]
        ax.text(x, i, f'  {abs(v):,}', va='center', fontsize=8)
    ax.set_yticks(range(len(f))); ax.set_yticklabels(labs, fontsize=8); ax.invert_yaxis()
    ax.set_title(SETS[k]['name']); ax.xaxis.set_visible(False)
    for s in ['top', 'right', 'bottom']: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## หมายเหตุ / ข้อจำกัด
- **นับแบบ "ไม่ตัดแถว" ใน L1–L2**: แถวที่เบอร์/อีเมลใช้ไม่ได้ยังอยู่ แค่ใช้คีย์นั้นจับคู่ไม่ได้
- **R6 ข้ามภาษา** ใช้แค่อักษรต้นชื่อ+นามสกุล + ต้องมีเบอร์/อีเมลร่วม → ยังไม่ใช่ transliteration เต็ม · ขั้นต่อไปใช้คู่ชื่อไทย-อังกฤษจาก ITOS (`S_CUSTOMER` 391k คน) สร้าง/ทดสอบกฎ
- **ชื่อปลอม** ใช้กฎ — ชื่อแปลกแต่เป็นคนจริงอาจโดนบ้าง (เช่นชื่อ 1 ตัวอักษร) · ไม่ลบทิ้ง แยกไว้ในถังของมันเพื่อตรวจได้
- **นิติบุคคล** แยกเป็นอีกชุด (dedup ด้วยชื่อบริษัท + เลขผู้เสียภาษีในเฟสถัดไป)
- **เลขบัตร** ใช้ได้เฉพาะ `member_auth.identity_no` · `customer_master.citizen_no` ถูกซ่อน ต้องขอสิทธิ์จากทีม Inno
- ตัวเลขเป็นของวันที่รัน · ข้อมูล CRM อัปเดต real-time